# Chapter 7 — Classes and Interfaces

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 7.

---


### Item 48 — Accept Functions Instead of Classes for Simple Interfaces

Python functions are first-class objects — many APIs that seem to call for a small class (a "strategy" or "callback" object with just one method) can instead just accept a plain function. `sorted(items, key=some_function)` is the canonical example: no `KeySelector` class or interface needed, any callable works.


In [1]:
def log_missing():
    print('key missing, returning default')
    return 0

# A plain function passed directly, no wrapper class needed
from collections import defaultdict
counts = defaultdict(log_missing)
counts['red'] += 1     # missing key -- log_missing() runs as the factory
print(dict(counts))

# sorted()'s key= accepts any callable -- no "Comparator" class required
words = ['banana', 'kiwi', 'apple', 'fig']
print(sorted(words, key=len))               # a builtin function works directly
print(sorted(words, key=lambda w: w[-1]))    # so does a lambda -- no class needed either


key missing, returning default
{'red': 1}
['fig', 'kiwi', 'apple', 'banana']
['banana', 'apple', 'fig', 'kiwi']


### Item 49 — Prefer Object-Oriented Polymorphism over Functions with `isinstance` Checks

A function with a chain of `isinstance(obj, TypeA)` / `isinstance(obj, TypeB)` checks to decide what to do is fragile — every new type requires editing that central function, and it's easy to forget a case. The object-oriented alternative — give each type its own method with the same name, and let normal method dispatch pick the right implementation — means adding a new type never requires touching existing code.


In [2]:
# isinstance-chain approach: adding a new shape means editing THIS function every time
class Circle:
    def __init__(self, radius):
        self.radius = radius

class Square:
    def __init__(self, side):
        self.side = side

def area_isinstance(shape):
    if isinstance(shape, Circle):
        return 3.14159 * shape.radius ** 2
    elif isinstance(shape, Square):
        return shape.side ** 2
    else:
        raise TypeError(f'Unknown shape: {type(shape)}')

print(area_isinstance(Circle(2)), area_isinstance(Square(3)))

# Polymorphism: each class knows how to compute its own area -- no central function to edit
class Shape:
    def area(self):
        raise NotImplementedError

class CircleP(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return 3.14159 * self.radius ** 2

class SquareP(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

shapes = [CircleP(2), SquareP(3)]
for s in shapes:
    print(s.area())   # correct method picked automatically -- adding TriangleP needs zero edits here


12.56636 9
12.56636
9


### Item 50 — Consider `functools.singledispatch` for Functional-Style Programming Instead of Object-Oriented Polymorphism

When you'd rather keep a function-based design (not every problem fits object-oriented dispatch cleanly, e.g. dispatching on *built-in* types you don't control) `@functools.singledispatch` gives function-style code the same "pick behavior by type" ability: define a generic fallback function, then register type-specific implementations with `@generic_func.register(SomeType)` — calling the generic name dispatches to the right one based on the first argument's type.


In [3]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f'a generic value: {value!r}'

@describe.register(int)
def _(value):
    return f'an integer: {value}'

@describe.register(list)
def _(value):
    return f'a list with {len(value)} items'

print(describe(42))          # dispatches to the int-specific version
print(describe([1, 2, 3]))   # dispatches to the list-specific version
print(describe(3.14))        # falls back to the generic version -- no float-specific registration


an integer: 42
a list with 3 items
a generic value: 3.14


### Item 51 — Prefer `dataclasses` for Defining Lightweight Classes

A plain class needing only to hold a handful of named fields (with `__init__`, `__repr__`, and `__eq__` all hand-written the same tedious way every time) is exactly what `@dataclass` automates: declare the fields with type annotations, and the decorator generates all of that boilerplate for you, correctly, every time.


In [4]:
# Hand-written: works, but every method is boilerplate you have to get right yourself
class PointManual:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f'PointManual(x={self.x!r}, y={self.y!r})'
    def __eq__(self, other):
        return isinstance(other, PointManual) and self.x == other.x and self.y == other.y

p1, p2 = PointManual(1, 2), PointManual(1, 2)
print(p1, p1 == p2)

from dataclasses import dataclass

@dataclass
class Point:          # __init__, __repr__, __eq__ all generated automatically from the annotations
    x: int
    y: int

p3, p4 = Point(1, 2), Point(1, 2)
print(p3, p3 == p4)


PointManual(x=1, y=2) True
Point(x=1, y=2) True


### Item 52 — Use `@classmethod` Polymorphism to Construct Objects Generically

When subclasses need their own custom construction logic (e.g., "build me from this file" or "build me from this config dict"), a `@classmethod` is the right tool: it receives the actual class it was called on (`cls`) — even through inheritance — so a base class can define one generic constructor pattern that automatically builds the correct subclass, without an `isinstance` check anywhere.


In [5]:
class Animal:
    def __init__(self, name):
        self.name = name

    @classmethod
    def from_config(cls, config):
        # cls is whichever class this was actually called on -- Animal, Dog, or Cat
        return cls(config['name'])

    def speak(self):
        return f'{self.name} makes a sound'

class Dog(Animal):
    def speak(self):
        return f'{self.name} barks'

class Cat(Animal):
    def speak(self):
        return f'{self.name} meows'

dog = Dog.from_config({'name': 'Rex'})     # cls is Dog here
cat = Cat.from_config({'name': 'Whiskers'}) # cls is Cat here -- SAME from_config code, different result
print(dog.speak())
print(cat.speak())
print(type(dog), type(cat))


Rex barks
Whiskers meows
<class '__main__.Dog'> <class '__main__.Cat'>


### Item 53 — Initialize Parent Classes with `super`

Always use `super().__init__(...)` to call a parent class's constructor, rather than naming the parent class explicitly (`ParentClass.__init__(self, ...)`). `super()` correctly follows the class's actual Method Resolution Order at runtime, which matters enormously the moment multiple inheritance is involved — hardcoding the parent's name breaks the moment the class hierarchy changes.


In [6]:
class Base:
    def __init__(self, value):
        print('Base.__init__')
        self.value = value

class Mixin:
    def __init__(self, *args, **kwargs):
        print('Mixin.__init__')
        super().__init__(*args, **kwargs)   # cooperates correctly with the MRO, whatever it is
        self.mixin_applied = True

class Combined(Mixin, Base):
    def __init__(self, value):
        super().__init__(value)   # follows the REAL MRO: Combined -> Mixin -> Base -> object
        print('Combined.__init__')

c = Combined(42)
print(c.value, c.mixin_applied)
print([cls.__name__ for cls in Combined.__mro__])   # this is the order super() actually follows


Mixin.__init__
Base.__init__
Combined.__init__
42 True
['Combined', 'Mixin', 'Base', 'object']


### Item 54 — Consider Composing Functionality with Mix-in Classes

A mix-in is a small class meant only to be combined with others via multiple inheritance — it provides one focused piece of reusable behavior (like "can be converted to a dict") and is never instantiated on its own. This lets you compose behavior from independent, reusable building blocks instead of building one deep, rigid inheritance chain.


In [7]:
class ToDictMixin:
    """Mix-in: adds generic dict-conversion to any class that uses it. Never used alone."""
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

class ComparableMixin:
    """Mix-in: adds equality comparison based on to_dict(), if available."""
    def __eq__(self, other):
        return isinstance(other, self.__class__) and self.to_dict() == other.to_dict()

class Employee(ToDictMixin, ComparableMixin):   # composes BOTH pieces of reusable behavior
    def __init__(self, name, role):
        self.name = name
        self.role = role

e1 = Employee('Ada', 'Engineer')
e2 = Employee('Ada', 'Engineer')
print(e1.to_dict())          # from ToDictMixin
print(e1 == e2)                # from ComparableMixin, built on top of ToDictMixin's method


{'name': 'Ada', 'role': 'Engineer'}
True


### Item 55 — Prefer Public Attributes over Private Ones

Python's `__double_underscore` attributes trigger name mangling (Item 4.2 in the PCAP track covers the mechanism) meant to prevent accidental clashes in subclasses — but the book's advice here is to reach for it rarely: prefer plain public attributes (or single-underscore `_protected` as a "internal use, but accessible" convention) since Python's culture favors trusting other programmers over hiding data behind hard-to-override private names, which can make legitimate subclassing awkward.


In [8]:
class MyClass:
    def __init__(self):
        self.public = 'anyone can use this'
        self._protected = 'convention: internal use, but still reachable if needed'
        self.__private = 'name-mangled -- awkward to override from a subclass'

    def get_private(self):
        return self.__private

obj = MyClass()
print(obj.public)
print(obj._protected)          # still accessible -- just a "please don't" convention
print(obj.get_private())
print(obj.__dict__)             # notice __private became _MyClass__private -- mangled, not truly hidden

class SubClass(MyClass):
    def __init__(self):
        super().__init__()
        # self.__private = 'trying to override'   # would create _SubClass__private, a DIFFERENT attribute!

print('Public/protected attributes compose far more predictably across subclasses.')


anyone can use this
convention: internal use, but still reachable if needed
name-mangled -- awkward to override from a subclass
{'public': 'anyone can use this', '_protected': 'convention: internal use, but still reachable if needed', '_MyClass__private': 'name-mangled -- awkward to override from a subclass'}
Public/protected attributes compose far more predictably across subclasses.


### Item 56 — Prefer `dataclasses` for Creating Immutable Objects

`@dataclass(frozen=True)` produces an object whose fields can't be reassigned after construction (attempting to raises `FrozenInstanceError`) — useful for values that should behave like something you can safely share and hash (frozen dataclasses are also hashable by default, ordinary ones aren't), without hand-writing `__setattr__` overrides yourself.


In [9]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    latitude: float
    longitude: float

point = Coordinate(40.7128, -74.0060)
print(point)

try:
    point.latitude = 0.0   # frozen=True blocks mutation after construction
except Exception as e:
    print(f'{type(e).__name__}:', e)

# Frozen dataclasses are hashable -- usable as dict keys or in sets, unlike a normal mutable dataclass
locations = {point: 'New York City'}
print(locations[Coordinate(40.7128, -74.0060)])   # equal value -> same hash -> found


Coordinate(latitude=40.7128, longitude=-74.006)
FrozenInstanceError: cannot assign to field 'latitude'
New York City


### Item 57 — Inherit from `collections.abc` Classes for Custom Container Types

Writing a custom list-like or dict-like class from scratch means implementing many dunder methods (`__len__`, `__getitem__`, `__iter__`, ...) just to behave correctly with `for`, `len()`, `in`, etc. Inheriting from the appropriate `collections.abc` class (`Sequence`, `Mapping`, `MutableSequence`, ...) means you only implement the *few* required abstract methods, and get all the standard container behaviors (iteration, containment, even mixins like `.index()`/`.count()`) for free.


In [10]:
from collections.abc import Sequence

class FrequencyList(Sequence):
    """A read-only sequence view that only requires __getitem__ and __len__ to be implemented."""
    def __init__(self, values):
        self._values = list(values)

    def __getitem__(self, index):
        return self._values[index]

    def __len__(self):
        return len(self._values)

freq = FrequencyList([10, 20, 30, 40])

print(len(freq))          # works: Sequence provides this via __len__
print(freq[1])             # works: via __getitem__
print(list(freq))           # iteration works too -- Sequence provides __iter__ automatically
print(20 in freq)           # __contains__ also comes free from Sequence
print(freq.index(30))       # .index() is a Sequence mixin method -- never implemented by us directly


4
20
[10, 20, 30, 40]
True
2
